<a href="https://colab.research.google.com/github/shubhampuri240/manisha-overseas-website/blob/staging/stage4/Manisha_Overseas_Free_TryOn_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manisha Overseas · Stage 4 — Free AI Try-On Test (Google Colab)

**Research / internal evaluation only. Do not deploy this as a customer-facing website feature.** This notebook tests FASHN VTON v1.5 by applying one Manisha Overseas catalogue garment to a separate **female fashion model photograph from your existing catalogue**. The demonstration person's photo is an original existing Manisha Overseas garment photograph; no new person image is generated or sourced. It does not upload private customer photos, expose any API key, or modify the live website.

### Quick start — use **Runtime → Run all** (do not click Save in GitHub)
1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another *free* GPU).
2. Select **Runtime → Run all** once; Colab will install the open-source package, download weights, display the selected dress and a separate Manisha Overseas female-model catalogue photograph, then attempt one image generation.
3. If you get a red error, send a screenshot showing **which section** failed. **Do not click "Save in GitHub":** the notebook is already saved in staging, and that popup is not related to running the code.
4. The GitHub file is a template: if you change `STYLE_CODE`, rerun the garment-selection cell and the generation cell (or choose Runtime → Run all).

### Important before running
- **Free access:** Colab's free GPU availability and quota are not guaranteed. Don't buy a plan to fix a Python error.
- **License:** Core FASHN VTON code/model is Apache-2.0; the standard pipeline loads `fashn-human-parser`, whose SegFormer dependency is restricted to **non-commercial research/evaluation**. Commercial website use isn't cleared.
- **Colab policy/privacy:** Use only catalogue fashion photographs for which your business has the necessary model releases/permission for AI outfit edits. The default image shows an adult female fashion model from Manisha Overseas style 3476 (or style 4638 if the selected garment is 3476). No private customers, minors, or other people's unapproved photos. Avoid Google Drive mounting and do not upload images to GitHub. Colab infrastructure processes this temporary environment.
- **GPU memory:** A T4 has enough memory for some models, but this pipeline uses float32 on older GPUs and can still run out of memory. If that happens, it does not imply a paid subscription is required.
- **Install:** Installing Python packages can take a few minutes. Read the green `INSTALLATION VERIFIED` message before continuing.

Official instructions: https://github.com/fashn-AI/fashn-vton-1.5  
Colab FAQ: https://research.google.com/colaboratory/faq.html

## 1 · Verify free GPU

This stops with a useful message if the runtime has no GPU. If you cannot choose a free GPU, you do not need to purchase anything; retry only when a free GPU is offered.

In [ ]:
import torch
from IPython.display import display, Markdown

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab: Runtime → Change runtime type → GPU. Free GPU availability is not guaranteed.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (gpu.total_memory / (1024**3)))
print("CUDA capability:", f"{gpu.major}.{gpu.minor}")
if gpu.major < 8:
    print("WARNING: This older GPU may need float32 and run out of VRAM. It may run out of GPU memory during generation; no purchase is necessary to diagnose this.")
display(Markdown("✅ GPU runtime detected. **No payment is required to run on the free tier if one is allocated.**"))

## 2 · Install and verify FASHN VTON (required)

Click ▶ and **wait for `INSTALLATION VERIFIED`**. This installs into the current Colab session. Do not proceed to Section 6 before Sections 2 and 3 finish.

In [ ]:
# One clean installation cell — run this before model weights or generation.
from pathlib import Path
import sys
import subprocess
import importlib

ROOT = Path("/content")
REPO = ROOT / "fashn-vton-1.5"
UPSTREAM = "https://github.com/fashn-AI/fashn-vton-1.5.git"

print("STEP 2 / Installing official open-source FASHN VTON package...")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", UPSTREAM, str(REPO)], check=True)
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError(
        f"Expected official project at {REPO}, but pyproject.toml is missing. "
        "Delete that incomplete folder in Colab Files and rerun this cell."
    )

# Install in the SAME Python environment used by this Colab notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", str(REPO)],
    check=True
)
# Editable installs sometimes need a kernel restart in Colab.
# Add the official src-layout path for this current notebook kernel.
source_dir = str(REPO / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
try:
    from fashn_vton import TryOnPipeline
except Exception as e:
    raise RuntimeError(
        "The package installation finished, but import still failed. "
        "Use Runtime → Restart session, then Runtime → Run all. "
        f"Original import error: {type(e).__name__}: {e}"
    ) from e
print("✅ INSTALLATION VERIFIED: fashn_vton imports correctly.")
print("Official project folder:", REPO)

## 3 · Download the model weights

The official download script obtains approximately **2 GB** of model and pose weights. Human-parser weights may download on first generation.

In [ ]:
# Run after STEP 2 completes successfully.
import sys
import subprocess
from pathlib import Path

if "REPO" not in globals() or not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("First run Section 2 — Install the official FASHN VTON repository.")

WEIGHTS = REPO / "weights"
required = [
    WEIGHTS / "model.safetensors",
    WEIGHTS / "dwpose" / "yolox_l.onnx",
    WEIGHTS / "dwpose" / "dw-ll_ucoco_384.onnx"
]
if not all(p.is_file() and p.stat().st_size > 0 for p in required):
    print("STEP 3 / Downloading official model and pose weights (large files)...")
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "download_weights.py"),
         "--weights-dir", str(WEIGHTS)],
        check=True
    )
missing = [str(p) for p in required if not (p.is_file() and p.stat().st_size > 0)]
if missing:
    raise RuntimeError("Weight download incomplete: " + ", ".join(missing))
print("✅ WEIGHTS VERIFIED. All three required files exist.")
print("Human parser weights may be fetched automatically during first inference.")

## 4 · Select an actual Manisha Overseas garment

Change `STYLE_CODE` to any code in the list. This fetches an **original catalogue image from GitHub's staging branch**; no photograph is recompressed or edited on GitHub. The model will resize its input internally.

In [ ]:
from io import BytesIO
import requests
from PIL import Image, ImageOps
from IPython.display import display

STYLES = {
    "4638": ("Floral printed long dress", "style-4638-front.webp"),
    "6821": ("Printed wide-leg one-piece", "style-6821-front.webp"),
    "3476": ("Printed long dress", "style-3476-front.webp"),
    "7284": ("Open-back long dress", "style-7284-front.webp"),
    "1509": ("Tiered long dress", "style-1509-front.webp"),
    "2941": ("Printed tiered dress", "style-2941-front.webp"),
    "9152": ("Green printed long dress", "style-9152-full.webp"),
}
STYLE_CODE = "4638"  # Change this to "3476", "9152", etc.

if STYLE_CODE not in STYLES:
    raise ValueError(f"Unknown style {STYLE_CODE}. Choose from: {', '.join(STYLES)}")
label, filename = STYLES[STYLE_CODE]
url = (
    "https://raw.githubusercontent.com/"
    "shubhampuri240/manisha-overseas-website/staging/assets/visuals/" + filename
)
with requests.get(url, timeout=45, stream=True) as response:
    response.raise_for_status()
    data = bytearray()
    for chunk in response.iter_content(chunk_size=65536):
        data.extend(chunk)
        if len(data) > 15 * 1024 * 1024:
            raise ValueError("Garment file larger than the 15 MB evaluation limit.")
with Image.open(BytesIO(data)) as im:
    if im.width * im.height > 30_000_000:
        raise ValueError("Garment photograph exceeds 30 megapixels.")
    garment = ImageOps.exif_transpose(im).convert("RGB").copy()
print(f"Selected Manisha Overseas style {STYLE_CODE}: {label}")
display(garment.resize((int(garment.width * min(420/garment.width, 1)), int(garment.height * min(420/garment.width, 1)))))

## 5 · Load an existing female fashion model photograph

The previous notebook used `examples/data/model.webp`, an official demo photograph featuring a male-presenting model. FASHN preserved that person's appearance when applying the dress.

**Updated default:** use the original Manisha Overseas Style **3476** female-model photograph as the person, and the garment you selected in Section 4 as the clothing reference. If you choose Style 3476 as the garment, the person photograph switches to Style **4638** so the model does not start wearing the same garment being tested.

This is an **internal evaluation only**. Before uploading/manipulating professional model photographs, verify that Manisha Overseas has permission to create AI-edited imagery featuring those models. Do not share generated likenesses publicly unless those rights and the model's commercial licensing are resolved.

In [ ]:
# Default test person: an existing female fashion model from our own catalogue.
# No upstream male demo person; no private/customer upload.
from io import BytesIO
from PIL import Image, ImageOps
import requests
from IPython.display import display

if "STYLE_CODE" not in globals() or "STYLES" not in globals():
    raise RuntimeError("Run Section 4 first to select a Manisha Overseas garment.")

PERSON_REFERENCE_STYLE = "4638" if STYLE_CODE == "3476" else "3476"
_, reference_filename = STYLES[PERSON_REFERENCE_STYLE]
person_url = (
    "https://raw.githubusercontent.com/"
    "shubhampuri240/manisha-overseas-website/staging/assets/visuals/"
    + reference_filename
)
with requests.get(person_url, timeout=45, stream=True) as response:
    response.raise_for_status()
    payload = bytearray()
    for chunk in response.iter_content(chunk_size=65536):
        payload.extend(chunk)
        if len(payload) > 15 * 1024 * 1024:
            raise ValueError("Model reference exceeds 15 MB.")
with Image.open(BytesIO(payload)) as im:
    if im.width * im.height > 30_000_000:
        raise ValueError("Model photograph exceeds 30 megapixels.")
    person = ImageOps.exif_transpose(im).convert("RGB").copy()

print(f"Female fashion-model catalogue reference: style {PERSON_REFERENCE_STYLE}")
print(f"Garment being applied: style {STYLE_CODE}")
print("Internal evaluation only: confirm appropriate model likeness/AI editing permissions.")
preview_scale = min(420 / person.width, 1)
display(person.resize((
    max(1, round(person.width * preview_scale)),
    max(1, round(person.height * preview_scale))
)))

## 6 · Generate the research-only AI try-on

After Sections 2–5 have completed, click ▶ here. The cell checks that the Python package, weights, garment and sample person are present. If the T4 runs out of memory, it will explain why. **No actual customer photographs and no commercial use.**

In [ ]:
# Run after Sections 2, 3, 4 and 5 have completed successfully.
import sys
from pathlib import Path
import torch
from IPython.display import display

ROOT = Path("/content")
REPO = ROOT / "fashn-vton-1.5"
WEIGHTS = REPO / "weights"
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Section 2 hasn't finished: run the installation cell first.")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
try:
    from fashn_vton import TryOnPipeline
except Exception as e:
    raise RuntimeError(
        "Cannot import fashn_vton. Run Section 2, confirm the green INSTALLATION VERIFIED message, "
        "and then retry. If imports still fail, restart session and Run all."
    ) from e

expected = [
    WEIGHTS / "model.safetensors",
    WEIGHTS / "dwpose" / "yolox_l.onnx",
    WEIGHTS / "dwpose" / "dw-ll_ucoco_384.onnx"
]
if any(not p.is_file() or p.stat().st_size == 0 for p in expected):
    raise RuntimeError("Section 3 is incomplete: download model weights first.")
if "person" not in globals() or "garment" not in globals():
    raise RuntimeError("Run Sections 4 and 5 to load the garment and the official demo person.")
if not torch.cuda.is_available():
    raise RuntimeError("No active GPU. Go to Runtime → Change runtime type → GPU.")

gpu_name = torch.cuda.get_device_name(0)
print("Using GPU:", gpu_name)
if not torch.cuda.is_bf16_supported():
    print("⚠️ This GPU does not support bf16, so the model will use float32 and may exceed VRAM.")
torch.cuda.empty_cache()
print("Initializing actual FASHN VTON v1.5 (research-only)...")
try:
    pipeline = TryOnPipeline(weights_dir=str(WEIGHTS), device="cuda")
    output = pipeline(
        person_image=person,
        garment_image=garment,
        category="one-pieces",
        garment_photo_type="model",
        num_samples=1,
        num_timesteps=20,
        seed=42,
        segmentation_free=True
    )
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        f"{gpu_name} ran out of GPU memory. The free T4 may be unable to run this model "
        "because older GPUs use float32. No paid upgrade is required to understand the error."
    ) from e

result = output.images[0]
RESULT_PATH = ROOT / ("manisha_tryon_" + STYLE_CODE + "_research.png")
result.save(RESULT_PATH)
print("✅ AI TRY-ON GENERATION COMPLETED. Internal evaluation only.")
display(result)
print("Temporary result:", RESULT_PATH)

## 7 · Compare the originals and generated preview

The output is a synthetic illustration, **not a size/fit guarantee**. Patterns, garment lengths and body details may be inaccurate.

In [ ]:
from IPython.display import display
print("Original catalogue garment:")
display(garment)
print("Female-model catalogue reference (existing Manisha Overseas photo):")
display(person)
print("AI-generated internal evaluation:")
display(result)

## 8 · Download your result (optional) and clear your temporary file

Use the output only to assess the model internally. Do **not** put generated person images on the public Manisha Overseas website or use them for ads without explicit model/likeness permissions and clearing the AI pipeline's commercial licensing. After evaluating, choose **Runtime → Disconnect and delete runtime** to remove the temporary VM; Google may still have separate operational retention.

In [ ]:
# Optional: uncomment only if you need a local copy for internal assessment.
# from google.colab import files
# files.download(str(RESULT_PATH))

print("Local result:", RESULT_PATH)
print("To remove the local result now, run:")
print("RESULT_PATH.unlink(missing_ok=True)")
print("Then use Runtime → Disconnect and delete runtime to discard the temporary VM.")